# VeloceAI — Colab Free (one-click launch)
1. اختَر GPU من Runtime → Change runtime type إذا كان متاحًا (اختياري؛ CPU أبطأ).
2. شغّل خلية التشغيل الوحيدة بالأسفل وانتظر ظهور رابط VeloceAI.
3. افتح الرابط واترك هذه الجلسة تعمل أثناء التحسين. الرابط مؤقت ويتوقف عند انتهاء جلسة Colab.

لا تحتاج Hugging Face أو اشتراكًا مدفوعًا. النموذج يُحمّل تلقائيًا عند أول عملية تحسين.

In [ ]:
import os, sys, time, re, pathlib, subprocess, urllib.request, json

def run(command):
    print('Running:', ' '.join(command[:5]), '...')
    subprocess.run(command, check=True)

# Install only the app dependencies; keep Colab's preinstalled PyTorch build.
run(['apt-get','-qq','update'])
run(['apt-get','-qq','install','-y','ffmpeg'])
run([sys.executable,'-m','pip','install','-q','fastapi','uvicorn[standard]','python-multipart','pillow','opencv-python-headless','spandrel'])

base='https://raw.githubusercontent.com/rezeyuri-dot/veloceai/main/'
urllib.request.urlretrieve(base+'colab_server.py','/content/colab_server.py')
urllib.request.urlretrieve(base+'index.html','/content/index.html')

# Stop old processes if this launch cell is run again.
for name in ('tunnel','server'):
    old=globals().get(name)
    try:
        if old and old.poll() is None:
            old.terminate(); old.wait(timeout=8)
    except Exception:
        pass

server_log=open('/content/veloceai-server.log','w')
server=subprocess.Popen([sys.executable,'/content/colab_server.py'],stdout=server_log,stderr=subprocess.STDOUT)
for _ in range(45):
    try:
        health=json.loads(urllib.request.urlopen('http://127.0.0.1:8000/health',timeout=3).read())
        if health.get('ok'): break
    except Exception: pass
    time.sleep(2)
else:
    print(pathlib.Path('/content/veloceai-server.log').read_text(errors='ignore')[-5000:])
    raise RuntimeError('VeloceAI server did not start. See the log above.')

cloudflared='/usr/local/bin/cloudflared'
if not pathlib.Path(cloudflared).exists():
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',cloudflared)
    os.chmod(cloudflared,0o755)
tunnel_log=open('/content/veloceai-tunnel.log','w')
tunnel=subprocess.Popen([cloudflared,'tunnel','--protocol','http2','--url','http://127.0.0.1:8000','--no-autoupdate'],stdout=tunnel_log,stderr=subprocess.STDOUT)
public_url=None
for _ in range(60):
    time.sleep(2)
    log=pathlib.Path('/content/veloceai-tunnel.log').read_text(errors='ignore')
    match=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',log)
    if match:
        public_url=match.group(0); break
if not public_url:
    print(pathlib.Path('/content/veloceai-tunnel.log').read_text(errors='ignore')[-5000:])
    raise RuntimeError('Could not obtain a temporary website URL.')

print('\n'+'='*60)
print('Local server health:',urllib.request.urlopen('http://127.0.0.1:8000/health',timeout=10).read().decode())
print('Loading AI model during startup (first launch may take a little longer)...')
try:
    warmup_request=urllib.request.Request('http://127.0.0.1:8000/warmup?profile=anime',data=b'',method='POST')
    warmup_result=json.loads(urllib.request.urlopen(warmup_request,timeout=600).read())
    print('Anime model ready:',warmup_result)
    standard_request=urllib.request.Request('http://127.0.0.1:8000/warmup?profile=general',data=b'',method='POST')
    standard_result=json.loads(urllib.request.urlopen(standard_request,timeout=600).read())
    print('General/Real model ready:',standard_result)
except Exception as e:
    raise RuntimeError('AI model warm-up failed: '+repr(e))

# Real API smoke tests on tiny synthetic media, including settings, video FPS and audio.
import io, json, requests, numpy as np
from PIL import Image, ImageDraw
from IPython.display import display, Video
test_dir=pathlib.Path('/content/veloceai-self-test')
test_dir.mkdir(exist_ok=True)
yy,xx=np.mgrid[0:64,0:64]
pattern=np.zeros((64,64,3),dtype=np.uint8)
pattern[:,:,0]=np.clip(xx*4,0,255)
pattern[:,:,1]=np.clip(yy*4,0,255)
pattern[:,:,2]=((xx//8%2)^(yy//8%2))*170+40
sample=Image.fromarray(pattern,'RGB')
draw=ImageDraw.Draw(sample)
draw.rectangle((12,12,51,51),outline=(255,255,255),width=2)
draw.ellipse((22,22,41,41),fill=(245,80,120))
image_buffer=io.BytesIO()
sample.save(image_buffer,format='PNG')
image_bytes=image_buffer.getvalue()

image_cases=[
    ('anime-2x-default',{'scale':2,'denoise':0,'sharpness':1.0,'profile':'anime','line_detail':20,'anti_alias':0}),
    ('anime-4x-max-controls',{'scale':4,'denoise':100,'sharpness':2.0,'profile':'anime','line_detail':100,'anti_alias':100}),
    ('general-2x',{'scale':2,'denoise':20,'sharpness':1.2,'profile':'general','line_detail':0,'anti_alias':10}),
    ('real-2x',{'scale':2,'denoise':10,'sharpness':1.15,'profile':'real','line_detail':0,'anti_alias':10})
]
for case_name,params in image_cases:
    case_started=time.perf_counter()
    response=requests.post('http://127.0.0.1:8000/api/enhance/image',
        params=params,files={'file':('veloceai-self-test.png',image_bytes,'image/png')},timeout=(15,600))
    if response.status_code!=200:
        raise RuntimeError(case_name+' failed ('+str(response.status_code)+'): '+response.text[:500])
    output=Image.open(io.BytesIO(response.content)).convert('RGB')
    expected=64*int(params['scale'])
    if output.size!=(expected,expected):
        raise RuntimeError(case_name+' returned '+str(output.size)+'; expected '+str((expected,expected)))
    pixels=np.asarray(output)
    if pixels.std()<2 or int(pixels.max())-int(pixels.min())<8:
        raise RuntimeError(case_name+' returned a blank or corrupted image')
    print('[PASS] '+case_name+': '+str(output.size[0])+'x'+str(output.size[1])+'; '+str(len(response.content))+' bytes; '+format(time.perf_counter()-case_started,'.1f')+'s')
    if case_name=='anime-4x-max-controls':
        sample.save(test_dir/'sample-input.png')
        output.save(test_dir/'sample-output-4x.png')
        print('Visual quality spot-check (synthetic input / AI output):')
        display(sample.resize((256,256)),output)

video_input=test_dir/'input-test.mp4'
video_output=test_dir/'output-test.mp4'
subprocess.run([
    'ffmpeg','-y','-hide_banner','-loglevel','error',
    '-f','lavfi','-i','testsrc2=size=64x64:rate=2:duration=1',
    '-f','lavfi','-i','sine=frequency=880:sample_rate=44100:duration=1',
    '-shortest','-c:v','libx264','-pix_fmt','yuv420p','-c:a','aac',str(video_input)
],check=True)
video_params={'scale':2,'denoise':0,'sharpness':1.0,'crf':28,
              'profile':'anime','line_detail':0,'anti_alias':0,'crop_black_bars':'true'}
video_started=time.perf_counter()
with open(video_input,'rb') as video_file:
    video_response=requests.post('http://127.0.0.1:8000/api/enhance/video',
        params=video_params,files={'file':('veloceai-self-test.mp4',video_file,'video/mp4')},
        timeout=(15,600))
if video_response.status_code!=200:
    raise RuntimeError('video test failed ('+str(video_response.status_code)+'): '+video_response.text[:700])
video_output.write_bytes(video_response.content)
probe=json.loads(subprocess.check_output(['ffprobe','-v','error','-show_streams','-show_format','-of','json',str(video_output)]))
video_streams=[st for st in probe.get('streams',[]) if st.get('codec_type')=='video']
audio_streams=[st for st in probe.get('streams',[]) if st.get('codec_type')=='audio']
if not video_streams or (video_streams[0].get('width'),video_streams[0].get('height'))!=(128,128):
    raise RuntimeError('video test returned an unexpected output resolution')
if not audio_streams:
    raise RuntimeError('video test failed to preserve its audio track')
fps=video_streams[0].get('avg_frame_rate','')
if fps!='2/1':
    raise RuntimeError('video test did not preserve 2 FPS; got '+str(fps))
print('[PASS] video: 128x128; 2 FPS preserved; audio preserved; '+str(len(video_response.content))+' bytes; '+format(time.perf_counter()-video_started,'.1f')+'s')
display(Video(str(video_output),embed=True))
print('[PASS] All VeloceAI image, video, and settings smoke tests passed.')
print('VeloceAI website:',public_url)
try:
    print('Public tunnel health:',urllib.request.urlopen(public_url+'/health',timeout=8).read().decode())
except Exception as e:
    print('The tunnel URL was created, but its public DNS check did not complete yet:',repr(e))
    print('Open the website URL above in your browser. If it does not open, wait 30 seconds and retry; if you use a VPN, temporarily turn it off or switch network.')
print('Keep this Colab cell/session running while using the website.')
print('='*60)
